# Checkpoint sweep eval

Runs `trainer.test(...)` (the same eval `scripts/test.py` runs for a single checkpoint) across every saved checkpoint in an experiment directory within an epoch range, and plots the metric trend (FGD, beat alignment, L1 diversity, facial L2/L-Vel) against epoch.

Unlike shelling out to `scripts/test.py` once per checkpoint, this builds **one** trainer instance (codecs + the big Moshi-embedding load from `get_model()` only happen once) and just swaps the GLM's own weights in place between checkpoints via `load_checkpoints`, which is the expensive part `test.py` would otherwise redo ~70 times.

Run this notebook on the same machine/env you launched training from (needs the GPU + the dataset caches).

In [ ]:
# --- config: edit these for your run ---

# Repo root -- all of config.py's relative paths (beatx_cache_path,
# upperbodycodec_ckpt, etc.) are written to be resolved against this, exactly
# like when you run `python scripts/test.py ...` from here on the CLI. The
# notebook kernel's own starting cwd is whatever Cursor/Jupyter launched it
# with (often your home dir, not the repo), so we chdir here explicitly in
# the next cell instead of assuming it.
REPO_ROOT = "/home/wenjye/miburi"

EXPERIMENT_DIR = "/home/wenjye/miburi/experiments/0807_1651_gtdm3_sharedregretrvq_global_scott_densefg_bf16_4_regret"
CONFIG_PATH = f"{EXPERIMENT_DIR}/config.yaml"

EPOCH_START = 1000
EPOCH_END = 2400
# None = every checkpoint found on disk in [EPOCH_START, EPOCH_END].
# Set to an int (e.g. 100) to subsample the sweep instead of hitting every
# saved checkpoint -- useful for a fast first pass before a full sweep.
EPOCH_STEP = None

# Same eval args as your manual `scripts/test.py` command. These paths are
# relative to REPO_ROOT, same as on the CLI.
EVAL_ARGS = [
    "--config", CONFIG_PATH,
    "--beatx_cache_path", "datasets/data_cache/beatx_gtdm3/database.hdf5",
    "--dataset_ratio", "scott_beatx_lowervalid",
    "--batch_size", "64",
    "--loader_workers", "8",
    "--eval_generation_mode", "production",
    "--cfg_scale", "1.5",
    "--random_seed", "2342",
]

# Cap batches per checkpoint for a faster exploratory sweep (None = full val/test set,
# same as a plain scripts/test.py run -- can be slow x N checkpoints).
MAX_BATCHES = None

CUDA_DEVICE = "0"
RESULTS_CSV = f"{EXPERIMENT_DIR}/checkpoint_sweep_{EPOCH_START}_{EPOCH_END}.csv"

In [ ]:
import os

# Must happen before anything reads a relative path (config.py's
# --beatx_cache_path/--upperbodycodec_ckpt/etc. defaults, and the config.yaml's
# own relative entries) -- this is what FileNotFoundError on
# experiments/demoexp_release_uppercodec/last_180.safetensors means: the
# kernel's cwd wasn't the repo root when that path got resolved.
os.chdir(REPO_ROOT)
assert os.path.isdir("experiments") and os.path.isdir("datasets"), (
    f"REPO_ROOT={REPO_ROOT!r} doesn't look like the miburi repo root "
    "(expected experiments/ and datasets/ subdirectories here)."
)

os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_DEVICE

import sys
import glob
import re
import warnings
warnings.simplefilter("ignore")

import pandas as pd
import matplotlib.pyplot as plt

# scripts/ must be on sys.path so `import trainers` resolves the same way
# scripts/test.py does when run from the repo root.
REPO_SCRIPTS_DIR = os.path.join(REPO_ROOT, "scripts")
if REPO_SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, REPO_SCRIPTS_DIR)

import trainers
from trainers.utils import config as cfgmod
from trainers.utils import tools as other_tools

In [ ]:
# --- discover checkpoints in range ---

ckpt_re = re.compile(r"last_(\d+)\.safetensors$")
available = {}
for path in glob.glob(os.path.join(EXPERIMENT_DIR, "last_*.safetensors")):
    m = ckpt_re.search(path)
    if m:
        available[int(m.group(1))] = path

epochs = sorted(e for e in available if EPOCH_START <= e <= EPOCH_END)
if EPOCH_STEP:
    epochs = [e for e in epochs if (e - epochs[0]) % EPOCH_STEP == 0]

if not epochs:
    raise RuntimeError(
        f"No checkpoints found in {EXPERIMENT_DIR} matching last_<epoch>.safetensors "
        f"within [{EPOCH_START}, {EPOCH_END}]."
    )
print(f"Found {len(epochs)} checkpoints to evaluate: {epochs[0]}..{epochs[-1]}")
print(epochs)

In [ ]:
# --- build the trainer once (loads codecs + Moshi embeddings + the first checkpoint) ---

sys.argv = ["checkpoint_sweep_eval"] + EVAL_ARGS + ["--test_ckpt", available[epochs[0]]]
args = cfgmod.parse_args()
args.is_train = False
args.ddp = False
args.name = os.path.dirname(args.test_ckpt)

other_tools.set_random_seed(args)
other_tools.print_exp_info(args)

trainer = getattr(trainers, args.trainer + "Trainer")(args)
print("Trainer built.")

In [ ]:
# --- sweep: swap weights in place, eval, record, checkpoint the CSV after every step ---

results = []
if os.path.exists(RESULTS_CSV):
    done = pd.read_csv(RESULTS_CSV)
    results = done.to_dict("records")
    already = set(done["epoch"].tolist())
    print(f"Resuming: {len(already)} epochs already in {RESULTS_CSV}, skipping those.")
else:
    already = set()

for epoch in epochs:
    if epoch in already:
        continue
    ckpt_path = available[epoch]
    print(f"\n=== epoch {epoch}: {ckpt_path} ===")
    try:
        other_tools.load_checkpoints(
            trainer.model, ckpt_path, rank=trainer.local_rank, after_distributed=False,
        )
        other_tools.set_random_seed(args) # same generation noise/CFG sampling per checkpoint
        metrics = trainer.test(
            f"sweep_{epoch}", visualize=False, max_batches=MAX_BATCHES, save=False,
        )
        metrics = dict(metrics)
        metrics["epoch"] = epoch
        results.append(metrics)
    except Exception as exc:
        print(f"FAILED at epoch {epoch}: {exc}")
        continue
    pd.DataFrame(results).sort_values("epoch").to_csv(RESULTS_CSV, index=False)

df = pd.DataFrame(results).sort_values("epoch").reset_index(drop=True)
df

In [ ]:
# --- plot the trend ---

panels = [
    ("fgd", None, "FGD (lower = closer to GT distribution)"),
    ("beat_alignment", "gt_beat_alignment", "Beat alignment (pred vs. GT)"),
    ("l1_diversity", "gt_l1_diversity", "L1 diversity (pred vs. GT)"),
    ("facial_l2", None, "Facial L2"),
    ("facial_lvel", None, "Facial L-Vel"),
    ("realtime_factor", None, "Sampling realtime factor"),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, (key, gt_key, title) in zip(axes.flat, panels):
    if key not in df.columns:
        ax.set_visible(False)
        continue
    ax.plot(df["epoch"], df[key], marker="o", label=key)
    if gt_key and gt_key in df.columns:
        ax.plot(df["epoch"], df[gt_key], marker="x", linestyle="--", label=gt_key)
        ax.legend(fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)

fig.tight_layout()
fig_path = os.path.join(EXPERIMENT_DIR, f"checkpoint_sweep_{EPOCH_START}_{EPOCH_END}.png")
fig.savefig(fig_path, dpi=150)
print(f"Saved plot to {fig_path}")
plt.show()